Script: this piece of code runs the SEOF package on single file input for 500mb geopotential height in the PNA and NAt regions
This is done for observations!!!

Output files: Standard EOFs for North Pacific & North Atlantic

PNA & NAT

SEOF, SPCS, FVAR, TVAR

For each model the z500 fields with and without the ensemble mean removed. 
_zgDJFerem
_zgDJF'

In [2]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 

In [3]:
#set up the data directory and load in lon + lat + time dimensions
outputdir2='/glade/work/nmaher/SEOF_output/'

model = 'OBS'
#option for single files - put one file path here to get lon/lat/time
ds_fx = xr.open_dataset(outputdir2+'hgt.mon.mean.1950-201512.nc_g025_500mb.nc')

lon = ds_fx.lon
lat = ds_fx.lat
time = ds_fx.time
zg_all=ds_fx.hgt

In [4]:
#select season
zg_DJF_full = zg_all.where(zg_all['time.season'] == 'DJF')

In [5]:
#take seasonal mean for masked and full regions
zg_DJF_full = zg_DJF_full.rolling(min_periods=3, center=True, time=3).mean()

In [6]:
# make annual mean
zg_DJF_full = zg_DJF_full.groupby('time.year').mean('time')

In [7]:
zg_DJF_full=np.squeeze(zg_DJF_full)

In [8]:
zg_DJF_full.shape

(66, 72, 144)

In [9]:
#remove the first time step as it is only JF not DJF
zg_DJF_2_full=zg_DJF_full[1:,:,:]
zg_DJF_2_full=zg_DJF_2_full.values

#DETREND HERE!
detrended=np.zeros([65,72,144])
from statsmodels.tsa.tsatools import detrend
for i in range(72):
    for j in range(144):
        detrended[:,i,j] = detrend(zg_DJF_2_full[:,i,j], order=2)


#remove the ensemble mean to get anomalies
zg_DJF_2e_full = detrended - np.ma.average(detrended,axis=0)
#zg_DJF_2e_fullb =  zg_DJF_2_full- np.ma.average(zg_DJF_2_full,axis=0)


In [10]:
#conda install -c conda-forge statsmodels


In [11]:
#mask the PNA region
lat_range=[20,90]
lon_range=[110,260]

lats=lat.values
lons=lon.values

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_or(lons<110,lons>260)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2e_full[0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2e_full[0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,:,:],zg_DJF_2e_full.shape)

masked_zgPNA=np.ma.masked_array(zg_DJF_2e_full,mask=mask4d)

In [12]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=3

#set up output dimensions
pna_seof = np.ma.masked_equal(np.zeros([neof,72,144]),0)
pna_spcs = np.ma.masked_equal(np.zeros([neof]),0)

pna_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([neof]),0)
pna_seof_totalVar_arr = np.ma.masked_equal(np.zeros([1]),0)
pna_eofs_northTest = np.ma.masked_equal(np.zeros([neof]),0)


zg_DJF_3 = masked_zgPNA.reshape(-1,72,144)

       
a=np.sqrt(coslat)
a2=a.values
wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

solver = Eof(zg_DJF_3, weights=wgts)


eofs = solver.eofs(neofs=neof, eofscaling=2)
pcs = solver.pcs(npcs=neof,pcscaling=1)
fracvar = solver.varianceFraction(neigs=neof)
total_variance = solver.totalAnomalyVariance()
    
pna_eofs_northTest = solver.northTest(neigs=neof, vfscaled=True)
    
spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
for j in range(len(pcs[0,:])):
    spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])



pna_seof = eofs
pna_spcs = spcs
pna_seof_fracVarExp_arr = fracvar
pna_seof_totalVar_arr = total_variance


In [13]:
#save output
eof_T='_PNA'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=pna_seof.data, mask=pna_seof.mask)
np.savez_compressed(outputdir2+model+eof_T+'_SPCS.npz', data=pna_spcs.data, mask=pna_spcs.mask)
np.savez_compressed(outputdir2+model+eof_T+'_FVAR.npz', data=pna_seof_fracVarExp_arr.data)
np.savez_compressed(outputdir2+model+eof_T+'_TVAR.npz', data=pna_seof_totalVar_arr.data)


In [14]:
#mask the NAt region
lat_range=[20,90]
lon_range=[280,10]

ilat=np.logical_or(lats<20,lats>90)
ilon = np.logical_and(lons>10,lons<280)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2e_full[0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2e_full[0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,:,:],zg_DJF_2e_full.shape)

masked_zgNAT=np.ma.masked_array(zg_DJF_2e_full,mask=mask4d)

In [15]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=1

#set up output dimensions
nat_seof = np.ma.masked_equal(np.zeros([neof,72,144]),0)
nat_spcs = np.ma.masked_equal(np.zeros([neof]),0)

nat_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([neof]),0)
nat_seof_totalVar_arr = np.ma.masked_equal(np.zeros([1]),0)
nat_eofs_northTest = np.ma.masked_equal(np.zeros([neof]),0)


#loop through each year and do the EOFs

zg_DJF_3 = masked_zgNAT.reshape(-1,72,144)

    
   
a=np.sqrt(coslat)
a2=a.values
wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

solver = Eof(zg_DJF_3, weights=wgts)


eofs = solver.eofs(neofs=neof, eofscaling=2)
pcs = solver.pcs(npcs=neof,pcscaling=1)
fracvar = solver.varianceFraction(neigs=neof)
total_variance = solver.totalAnomalyVariance()
    
nat_eofs_northTest = solver.northTest(neigs=neof, vfscaled=True)
    
spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
for j in range(len(pcs[0,:])):
    spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])


nat_seof= eofs
nat_spcs= spcs
nat_seof_fracVarExp_arr = fracvar
nat_seof_totalVar_arr = total_variance


In [16]:
#save output
eof_T='_NAT'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=nat_seof.data, mask=nat_seof.mask)
np.savez_compressed(outputdir2+model+eof_T+'_SPCS.npz', data=nat_spcs.data, mask=nat_spcs.mask)
np.savez_compressed(outputdir2+model+eof_T+'_FVAR.npz', data=nat_seof_fracVarExp_arr.data)
np.savez_compressed(outputdir2+model+eof_T+'_TVAR.npz', data=nat_seof_totalVar_arr.data)

In [17]:
#save DJF emeanremoved zg and zg
eof_T='_zgDJFerem'
np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=zg_DJF_2e_full.data, mask=zg_DJF_2e_full.mask)


eof_T='_zgDJF'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=zg_DJF_2_full.data, mask=zg_DJF_2e_full.mask)
